# NHDPlus Data Exploration

This notebook inventories the supplied NHDPlus V2.1 Great Lakes region 04 files and previews the main inputs used to build a stream network and relate it to drainage areas.

Raw inputs are under `../data/raw/nhdplus/`. Treat everything there as read-only; this notebook only reads the source files.

The inspection includes catchment-layer feature count, geometry type, CRS, bounds, fields, a map of up to 5,000 catchments, and sample attribute rows. It also previews connectivity (`PlusFlow`), flowline attributes (`PlusFlowlineVAA`), and EROM flow estimates. The map uses a capped feature preview to keep rendering responsive.

In [6]:
# Check if the NHDPlus files needed for the river network are available.

from pathlib import Path
import geopandas as gpd
import pyogrio

RAW = Path("../data/raw/nhdplus")

datasets = {
    "Connectivity": RAW / "NHDPlusAttribute/NHDPlus04/NHDPlusAttributes/PlusFlow.dbf",
    "Flowline attributes": RAW / "NHDPlusAttribute/NHDPlus04/NHDPlusAttributes/PlusFlowlineVAA.dbf",
    "Catchments": RAW / "NHDPlusCatchments/NHDPlus04/NHDPlusCatchment/Catchment.shp",
    "NHD geodatabase": RAW / "NHDSnapshotFGDB/NHDPlus04/NHDSnapshot/NHDSnapshot.gdb",
}

for name, path in datasets.items():
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

Connectivity: OK
Flowline attributes: OK
Catchments: OK
NHD geodatabase: OK


In [7]:
# Look at the PlusFlow table to see how river reaches are connected.

plusflow = gpd.read_file(datasets["Connectivity"], rows=5)

display(plusflow.drop(columns="geometry", errors="ignore"))

,FROMCOMID,FROMHYDSEQ,FROMLVLPAT,TOCOMID,TOHYDSEQ,TOLVLPAT,NODENUMBER,DELTALEVEL,DIRECTION,GAPDISTKM,HasGeo,TotDASqKM,DivDASqKM
0,1763922,90073479,90029498,1764194,90037496,90029498,90000541,0,709,0.0,Y,45.1593,45.1593
1,0,0,0,1764208,90107443,90029498,90073360,0,712,0.0,Y,0.0000,0.0000
2,1764208,90107443,90029498,1763922,90073479,90029498,90000532,0,709,0.0,Y,34.1982,34.1982
3,1776784,90056880,90047678,1777620,90047678,90047678,90001935,0,709,0.0,Y,16.3341,16.3341
4,1777620,90047678,90047678,1777628,90005554,90004432,90001939,1,709,0.0,Y,16.4214,16.4214


In [8]:
# Check the available attributes for each river reach.

vaa = gpd.read_file(datasets["Flowline attributes"], rows=5)

display(vaa.drop(columns="geometry", errors="ignore"))

,ComID,Fdate,StreamLeve,StreamOrde,StreamCalc,FromNode,ToNode,Hydroseq,LevelPathI,Pathlength,...,DivEffect,VPUIn,VPUOut,AreaSqKM,TotDASqKM,DivDASqKM,Tidal,TOTMA,WBAreaType,PathTimeMA
0,4799598.0,2011-08-29,4.0,1.0,1.0,90076181.0,90006588.0,90104691.0,90072107.0,18.033,...,0.0,0.0,0.0,0.3708,0.371,0.371,0.0,21.307158,LakePond,64.894333
1,4797518.0,2011-08-29,3.0,1.0,1.0,90075968.0,90006592.0,90104885.0,90104885.0,17.473,...,0.0,0.0,0.0,2.1771,2.177,2.177,0.0,0.125188,NaN,64.865177
2,4797542.0,2011-08-29,3.0,1.0,1.0,90075970.0,90006595.0,90104883.0,90104883.0,18.485,...,0.0,0.0,0.0,4.8915,4.891,4.891,0.0,0.119158,NaN,64.918933
3,4798436.0,2011-08-29,3.0,2.0,2.0,90006899.0,90006902.0,90072031.0,90072031.0,12.174,...,0.0,0.0,0.0,3.6954,14.625,14.625,0.0,0.142159,NaN,0.309131
4,4799656.0,2011-08-29,3.0,1.0,1.0,90076210.0,90006670.0,90104662.0,90047261.0,9.576,...,0.0,0.0,0.0,5.1309,5.131,5.131,0.0,1.913280,LakePond,2.465796


In [9]:
# Check the number of catchments and the fields available in the dataset.

info = pyogrio.read_info(datasets["Catchments"])

print("Catchments:", info["features"])
print("CRS:", info["crs"])
print("Fields:", info["fields"])

Catchments: 105849
CRS: EPSG:4269
Fields: ['GRIDCODE' 'FEATUREID' 'SOURCEFC' 'AreaSqKM']


In [11]:
# Check the river flowline layer and preview its main attributes.

flowline_info = pyogrio.read_info(
    datasets["NHD geodatabase"],
    layer="NHDFlowline"
)

print("Total flowlines:", flowline_info["features"])
print("Geometry:", flowline_info["geometry_type"])
print("CRS:", flowline_info["crs"])
print("Available fields:", flowline_info["fields"])

flowlines = gpd.read_file(
    datasets["NHD geodatabase"],
    layer="NHDFlowline",
    rows=5
)

display(flowlines.drop(columns="geometry").head())

Total flowlines: 114041
Geometry: MultiLineString Z
CRS: EPSG:4269
Available fields: ['Permanent_Identifier' 'FDate' 'Resolution' 'GNIS_ID' 'GNIS_Name'
 'LengthKM' 'ReachCode' 'FlowDir' 'WBArea_Permanent_Identifier' 'FType'
 'FCode' 'Shape_Length']


/home/rasheedr/graph-based-water-quality/.venv/lib/python3.12/site-packages/pyogrio/core.py:320: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_read_info(
/home/rasheedr/graph-based-water-quality/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_read(


,Permanent_Identifier,FDate,Resolution,GNIS_ID,GNIS_Name,LengthKM,ReachCode,FlowDir,WBArea_Permanent_Identifier,FType,FCode,Shape_Length
0,21978951,1999-09-29,3,,,2.671,02050102007273,0,0,460,46006,0.026426
1,22026534,1999-09-29,3,,,0.784,02050102007274,0,0,460,46006,0.007588
2,22026536,1999-09-29,3,,,3.276,02050102007275,0,0,460,46006,0.031230
3,21984553,1999-09-29,3,,,0.033,02050103006779,0,0,460,46006,0.000403
4,15547779,1999-09-29,3,,,2.213,02050105004379,0,0,460,46006,0.021472
